In [1]:
import os
import numpy as np
import torch
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [2]:
# ============================================================
# CARGA DEL DATASET
# ============================================================
AÑO = 2017
data = torch.load(
    f"/home/angelo/MyStore/Proyecto Investigador/{AÑO}/dataset_split_{AÑO}/dataset_split_{AÑO}_RF_(backscatter)_v1.pt",
    weights_only=False
)
train_data = data["train"]
test_data  = data["test"]

# ============================================================
# DICCIONARIO MAESTRO (todas las clases posibles del proyecto)
# Fijo para todos los años — los códigos nunca cambian
# ============================================================
CLASES_MAESTRO = {
    "Alfalfa":     1,
    "Barley":      2,
    "Carrot":      3,
    "Chickpea":    4,
    "Cotton":      5,
    "Fallow":      6,
    "Maize":       7,
    "Oats":        8,
    "Onion":       9,
    "Pepper":      10,
    "Potato":      11,
    "Pumpkin":     12,
    "Quinoa":      13,
    "Rice":        14,
    "SugarBeet":   15,
    "Sunflower":   16,
    "SweetPotato": 17,
    "Tomato":      18,
    "Wheat":       19
}

# ============================================================
# FILTRO AUTOMÁTICO: solo lo que está en el dataset de este año
# No hay exclusiones manuales — el dataset ya es la verdad
# ============================================================
clases_disponibles_train = set(train_data.keys())
clases_disponibles_test  = set(test_data.keys())

# Solo clases presentes en AMBOS splits
clases_cultivo = {
    nombre: codigo
    for nombre, codigo in CLASES_MAESTRO.items()
    if nombre in clases_disponibles_train
    and nombre in clases_disponibles_test
}

# ============================================================
# DIAGNÓSTICO
# ============================================================
clases_solo_train = clases_disponibles_train - clases_disponibles_test
clases_solo_test  = clases_disponibles_test  - clases_disponibles_train
clases_ausentes   = (
    set(CLASES_MAESTRO.keys())
    - clases_disponibles_train
    - clases_disponibles_test
)

print("=" * 55)
print(f"Año del dataset  : {AÑO}")
print(f"Clases activas   : {len(clases_cultivo)}")
print(f"Ausentes en ambos: {sorted(clases_ausentes)}")
print(f"Solo en train    : {sorted(clases_solo_train)}")
print(f"Solo en test     : {sorted(clases_solo_test)}")
print("=" * 55)
for nombre, codigo in sorted(clases_cultivo.items(), key=lambda x: x[1]):
    print(f"  {codigo:>3}  {nombre}")

# ============================================================
# CONSTRUIR X_train / Y_train  (solo clases activas)
# ============================================================
X_train_list, Y_train_list = [], []

for c, label in clases_cultivo.items():

    if c not in train_data:
        print(f"⚠ {c} no está en train_data, se omite.")
        continue

    X = train_data[c]["X"]
    if isinstance(X, torch.Tensor):
        X = X.numpy()
    if X.ndim == 3:                       # (N, 2, 61) → (N, 122)
        X = X.reshape(X.shape[0], -1)

    X_train_list.append(X)
    Y_train_list.append(np.full(len(X), label))

X_train = np.concatenate(X_train_list, axis=0)
Y_train = np.concatenate(Y_train_list, axis=0)

print(f"\nX_train shape : {X_train.shape}")
print(f"Y_train shape : {Y_train.shape}")
print(f"Clases        : {np.unique(Y_train).tolist()}")

# ============================================================
# CONSTRUIR X_test / Y_test  (solo clases activas)
# ============================================================
X_test_list, Y_test_list = [], []

for c, label in clases_cultivo.items():

    if c not in test_data:
        print(f"⚠ {c} no está en test_data, se omite.")
        continue

    X = test_data[c]["X"]
    if isinstance(X, torch.Tensor):
        X = X.numpy()
    if X.ndim == 3:
        X = X.reshape(X.shape[0], -1)

    X_test_list.append(X)
    Y_test_list.append(np.full(len(X), label))

X_test = np.concatenate(X_test_list, axis=0)
Y_test = np.concatenate(Y_test_list, axis=0)

print(f"\nX_test shape  : {X_test.shape}")
print(f"Y_test shape  : {Y_test.shape}")



Año del dataset  : 2017
Clases activas   : 16
Ausentes en ambos: ['Barley', 'Oats', 'Rice']
Solo en train    : []
Solo en test     : []
    1  Alfalfa
    3  Carrot
    4  Chickpea
    5  Cotton
    6  Fallow
    7  Maize
    9  Onion
   10  Pepper
   11  Potato
   12  Pumpkin
   13  Quinoa
   15  SugarBeet
   16  Sunflower
   17  SweetPotato
   18  Tomato
   19  Wheat

X_train shape : (21776, 122)
Y_train shape : (21776,)
Clases        : [1, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 15, 16, 17, 18, 19]

X_test shape  : (568967, 122)
Y_test shape  : (568967,)


In [3]:
# ============================================================
# ENTRENAMIENTO
# ============================================================
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=10,
    min_samples_leaf=4,
    class_weight="balanced_subsample",
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, Y_train)
print("\nRF entrenado correctamente.")




RF entrenado correctamente.


In [4]:
# ============================================================
# GUARDAR MODELO + METADATOS
# ============================================================
RUTA_MODELOS_DIR = f"/home/angelo/MyStore/Proyecto Investigador/{AÑO}/modelos_RF/"
os.makedirs(RUTA_MODELOS_DIR, exist_ok=True)

RUTA_MODELO = f"{RUTA_MODELOS_DIR}rf_modelo_(backscatter)_v1.joblib"

joblib.dump(
    {
        "model":          rf,
        "clases_cultivo": clases_cultivo,
        "input_shape":    X_train.shape[1],
        "año":            AÑO,
    },
    RUTA_MODELO
)

print(f"\nModelo RF guardado en:\n   {RUTA_MODELO}")


Modelo RF guardado en:
   /home/angelo/MyStore/Proyecto Investigador/2017/modelos_RF/rf_modelo_(backscatter)_v1.joblib
